In [1]:
!pip install rank_bm25 pyvi sentence-transformers --break-system-packages
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
import json, os, glob, re, gc, torch
from sentence_transformers import SentenceTransformer, util, CrossEncoder
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
from pyvi import ViTokenizer
from collections import defaultdict

CACHE_DIR = "/kaggle/working/cache"
os.makedirs(CACHE_DIR, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 65.0 MB/s eta 0:00:00


In [2]:
def extract_title(passage, first_dieu_pos):
    """Lấy tiêu đề văn bản (nằm giữa loại văn bản và trước dòng 'Căn cứ')"""
    preamble = passage[:first_dieu_pos]
    match = re.search(
        r'(THÔNG TƯ|NGHỊ ĐỊNH|QUYẾT ĐỊNH|LUẬT|THÔNG TƯ LIÊN TỊCH)\s*\n+(.*?)(?=Căn cứ|Theo đề nghị|$)',
        preamble, re.DOTALL
    )
    if match:
        title = match.group(2).replace("\r\n", " ").replace("\n", " ")
        title = re.sub(r'\s+', ' ', title).strip()
        return f"{match.group(1)}: {title}"
    return None


def chunk_document(passage, doc_id, max_words=200, overlap=30, include_appendix=True):
    """
    Tách 1 văn bản luật dài thành nhiều chunk nhỏ, GIỮ LẠI parent doc_id
    để khi retrieval trả về chunk nào, vẫn map lại đúng document gốc.
    Mỗi chunk được gắn kèm TIÊU ĐỀ văn bản ở đầu (giúp BM25 bắt từ khóa tên luật).
    include_appendix=False -> bỏ hẳn phụ lục (giảm mạnh số lượng chunk, tăng tốc độ).
    """
    dieu_positions = [m.start() for m in re.finditer(r'Điều\s+\d+\.', passage)]
    if not dieu_positions:
        return []

    title = extract_title(passage, dieu_positions[0])

    phuluc_match = re.search(r'PHỤ LỤC', passage)
    body_end = phuluc_match.start() if phuluc_match else len(passage)
    body = passage[dieu_positions[0]:body_end]  # bỏ hẳn phần thể thức hành chính trước Điều 1

    dieu_positions_body = [m.start() for m in re.finditer(r'Điều\s+\d+\.', body)]
    dieu_positions_body.append(len(body))

    chunks = []
    for i in range(len(dieu_positions_body) - 1):
        seg = body[dieu_positions_body[i]:dieu_positions_body[i+1]].strip()
        if seg:
            text_with_title = f"{title}\n{seg}" if title else seg
            chunks.append({"doc_id": doc_id, "type": "dieu", "text": text_with_title})

    if include_appendix and phuluc_match:
        appendix = passage[phuluc_match.start():]
        pl_positions = [m.start() for m in re.finditer(r'PHỤ LỤC[^\n]{0,40}', appendix)]
        pl_positions.append(len(appendix))
        for i in range(len(pl_positions) - 1):
            seg = appendix[pl_positions[i]:pl_positions[i+1]].strip()
            if not seg:
                continue
            words = seg.split()
            if len(words) <= max_words:
                text_with_title = f"{title}\n{seg}" if title else seg
                chunks.append({"doc_id": doc_id, "type": "phu_luc", "text": text_with_title})
            else:
                for start in range(0, len(words), max_words - overlap):
                    window = " ".join(words[start:start + max_words])
                    text_with_title = f"{title}\n{window}" if title else window
                    chunks.append({"doc_id": doc_id, "type": "phu_luc", "text": text_with_title})
    return chunks

In [3]:
CONTEXT_DIR = "/kaggle/input/datasets/thnhnguynchtin/legalir/selected-contexts/selected-contexts"
INCLUDE_APPENDIX = True

corpus_cache_path = f"{CACHE_DIR}/corpus_{'with' if INCLUDE_APPENDIX else 'no'}_appendix.json"

if os.path.exists(corpus_cache_path):
    print("Đang load corpus từ cache...")
    cached = json.load(open(corpus_cache_path, encoding="utf-8"))
    corpus_texts, corpus_doc_ids = cached["texts"], cached["doc_ids"]
else:
    print("Đang build corpus từ file gốc...")
    all_chunks = []
    for fpath in tqdm(sorted(glob.glob(os.path.join(CONTEXT_DIR, "*.json"))), desc="Chunking documents"):
        with open(fpath, encoding="utf-8") as f:
            data = json.load(f)
        doc_chunks = chunk_document(data["passage"], str(data["id"]), include_appendix=INCLUDE_APPENDIX)
        all_chunks.extend(doc_chunks)

    corpus_texts = [c["text"] for c in all_chunks]
    corpus_doc_ids = [c["doc_id"] for c in all_chunks]

    json.dump({"texts": corpus_texts, "doc_ids": corpus_doc_ids},
               open(corpus_cache_path, "w", encoding="utf-8"))

print(f"Số document gốc: {len(glob.glob(os.path.join(CONTEXT_DIR, '*.json')))}")
print(f"Số chunk trong corpus: {len(corpus_texts)}")

docid_to_chunk_idxs = defaultdict(list)
for idx, doc_id in enumerate(corpus_doc_ids):
    docid_to_chunk_idxs[doc_id].append(idx)

Đang build corpus từ file gốc...


Chunking documents: 100%|██████████| 8532/8532 [02:24<00:00, 58.93it/s]


Số document gốc: 8532
Số chunk trong corpus: 215461


In [4]:
with open("/kaggle/input/datasets/thnhnguynchtin/legalir/train.json", encoding="utf-8") as f:
    train_data = json.load(f)

questions_train = [v["question"] for v in train_data.values()]
golds_docid_sets = [set(v["answer"]) for v in train_data.values()]

print(f"Số câu hỏi train: {len(questions_train)}")

Số câu hỏi train: 7000


In [5]:
def ranked_chunks_to_docids(ranked_chunk_ids, top_k=None):
    """Chuyển danh sách chunk đã rank thành danh sách doc_id duy nhất, giữ thứ tự."""
    seen = set()
    doc_ids = []
    for cid in ranked_chunk_ids:
        doc_id = corpus_doc_ids[cid]
        if doc_id not in seen:
            seen.add(doc_id)
            doc_ids.append(doc_id)
        if top_k is not None and len(doc_ids) >= top_k:
            break
    return doc_ids


def precision_recall_f2(predicted_docids, gold_docids, beta=2):
    pred_set = set(predicted_docids)
    if not pred_set:
        return 0.0, 0.0, 0.0
    tp = len(pred_set & gold_docids)
    precision = tp / len(pred_set)
    recall = tp / len(gold_docids) if gold_docids else 0.0
    if precision + recall == 0:
        return precision, recall, 0.0
    f2 = (1 + beta**2) * precision * recall / (beta**2 * precision + recall)
    return precision, recall, f2

In [6]:
def tokenize_doc(text: str):
    segmented_text = ViTokenizer.tokenize(str(text))
    text_clean = re.sub(r"[^\w\s]", " ", segmented_text)
    return text_clean.lower().split()

tokenize = tokenize_doc  # tên dùng chung cho cả câu hỏi lẫn corpus

tokenized_cache_path = f"{CACHE_DIR}/tokenized_corpus_{'with' if INCLUDE_APPENDIX else 'no'}_appendix.json"

if os.path.exists(tokenized_cache_path):
    print("Đang load tokenized corpus từ cache...")
    tokenized_corpus = json.load(open(tokenized_cache_path, encoding="utf-8"))
else:
    print(f"Đang tokenize {len(corpus_texts)} chunk...")
    with ProcessPoolExecutor() as executor:
        tokenized_corpus = list(
            tqdm(executor.map(tokenize_doc, corpus_texts, chunksize=50),
                 total=len(corpus_texts), desc="Tokenizing")
        )
    json.dump(tokenized_corpus, open(tokenized_cache_path, "w", encoding="utf-8"))

bm25 = BM25Okapi(tokenized_corpus)
print("Đã build xong BM25 index.")

Đang tokenize 215461 chunk...


Tokenizing: 100%|██████████| 215461/215461 [11:24<00:00, 314.88it/s]


Đã build xong BM25 index.


In [7]:
dense_model = SentenceTransformer('AITeamVN/Vietnamese_Embedding', device='cuda')
dense_model.max_seq_length = 256

embeddings_cache_path = f"{CACHE_DIR}/corpus_embeddings_{'with' if INCLUDE_APPENDIX else 'no'}_appendix.pt"

if os.path.exists(embeddings_cache_path):
    print("Đang load corpus embeddings từ cache...")
    corpus_embeddings = torch.load(embeddings_cache_path)
else:
    print("Đang encode corpus...")
    corpus_embeddings = dense_model.encode(
        corpus_texts, batch_size=256, convert_to_tensor=True,
        show_progress_bar=True, device='cuda'
    )
    torch.save(corpus_embeddings, embeddings_cache_path)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/708 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Đang encode corpus...


Batches:   0%|          | 0/842 [00:00<?, ?it/s]

In [8]:
def compute_batch_scores(q_batch, encode_batch_size=64):
    """
    Tính điểm BM25 + Dense cho 1 lô câu hỏi.
    Dùng chung cho evaluate_alpha_batched và bất kỳ bước đánh giá nào khác cần điểm số thô.
    Trả về (dense_batch, bm25_batch) shape: (len(q_batch), n_corpus)
    """
    dense_batch = util.cos_sim(
        dense_model.encode(q_batch, convert_to_tensor=True,
                            batch_size=encode_batch_size, show_progress_bar=False),
        corpus_embeddings
    ).cpu().numpy().astype(np.float32)

    bm25_batch = np.array(
        [bm25.get_scores(tokenize(q)).astype(np.float32) for q in q_batch],
        dtype=np.float32
    )
    return dense_batch, bm25_batch

In [9]:
def evaluate_alpha_batched(questions, golds_docid_sets, alphas, ks=(1, 5, 10),
                            encode_batch_size=64, question_chunk_size=200, hybrid_top_n=50):
    n = len(questions)
    max_k = max(ks)

    hits_per_alpha = {alpha: {k: 0 for k in ks} for alpha in alphas}
    f2_sum_per_alpha = {alpha: 0.0 for alpha in alphas}
    ranks_per_alpha = {alpha: [] for alpha in alphas}

    print(f"Đang tính Hybrid Search theo lô ({question_chunk_size} câu/lần)...")

    for start in tqdm(range(0, n, question_chunk_size), desc="Question chunks"):
        end = min(start + question_chunk_size, n)
        q_batch = questions[start:end]
        gold_batch = golds_docid_sets[start:end]

        dense_batch = util.cos_sim(
            dense_model.encode(q_batch, convert_to_tensor=True,
                                batch_size=encode_batch_size, show_progress_bar=False),
            corpus_embeddings
        )  # tensor GPU, shape (batch, n_corpus)

        bm25_batch_np = np.array(
            [bm25.get_scores(tokenize(q)).astype(np.float32) for q in q_batch],
            dtype=np.float32
        )
        bm25_batch = torch.tensor(bm25_batch_np, device='cuda')

        bm25_norm = bm25_batch / (bm25_batch.max(dim=1, keepdim=True).values + 1e-9)
        dense_norm = dense_batch / (dense_batch.max(dim=1, keepdim=True).values + 1e-9)

        for alpha in alphas:
            final_scores = alpha * dense_norm + (1 - alpha) * bm25_norm

            topk_vals, topk_idx = torch.topk(final_scores, hybrid_top_n, dim=1)
            topk_idx_np = topk_idx.cpu().numpy()
            for i in range(len(q_batch)):
                ranked_chunks = topk_idx_np[i]
                ranks_per_alpha[alpha].append(ranked_chunks)

                doc_ids_ranked = ranked_chunks_to_docids(ranked_chunks, top_k=max_k)
                gold = gold_batch[i]
                for k in ks:
                    if set(doc_ids_ranked[:k]) & gold:
                        hits_per_alpha[alpha][k] += 1
                _, _, f2 = precision_recall_f2(doc_ids_ranked[:10], gold)
                f2_sum_per_alpha[alpha] += f2

        del dense_batch, bm25_batch, bm25_norm, dense_norm
        torch.cuda.empty_cache()
        gc.collect()

    print(f"\n{'Alpha':<8} " + " ".join(f"{'Hit@'+str(k):>10}" for k in ks) + f" {'F2@10':>10}")
    best_alpha, best_f2 = None, -1
    for alpha in alphas:
        result = {k: hits_per_alpha[alpha][k] / n for k in ks}
        f2_avg = f2_sum_per_alpha[alpha] / n
        print(f"{alpha:<8} " + " ".join(f"{result[k]*100:>9.2f}%" for k in ks) + f" {f2_avg:>9.4f}")
        if f2_avg > best_f2:
            best_f2 = f2_avg
            best_alpha = alpha

    print(f"\n>>> Alpha tốt nhất theo F2@10: alpha = {best_alpha} (F2 = {best_f2:.4f})")
    return best_alpha, ranks_per_alpha[best_alpha]

In [10]:
reranker = CrossEncoder('AITeamVN/Vietnamese_Reranker', max_length=256, device='cuda')

def evaluate_reranker(questions, hybrid_chunk_ranks, golds_docid_sets,
                       final_top_k=10, ks=(1, 5, 10), rerank_batch_size=64):
    n = len(questions)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Đang chuẩn bị cặp để rerank...")
    all_pairs, pair_owner = [], []
    for i, ranks in enumerate(hybrid_chunk_ranks):
        for cid in ranks:
            all_pairs.append([questions[i], corpus_texts[cid]])
            pair_owner.append(i)

    print(f"Tổng số cặp cần rerank: {len(all_pairs)}")
    all_scores = reranker.predict(all_pairs, batch_size=rerank_batch_size, show_progress_bar=True)

    hits_hybrid = {k: 0 for k in ks}
    hits_rerank = {k: 0 for k in ks}
    f2_hybrid_sum, f2_rerank_sum = 0.0, 0.0

    idx = 0
    for i in range(n):
        ranks = hybrid_chunk_ranks[i]
        m = len(ranks)
        scores = all_scores[idx: idx + m]
        idx += m

        reranked_chunks = [cid for _, cid in sorted(zip(scores, ranks), key=lambda x: -x[0])]

        gold = golds_docid_sets[i]
        docids_hybrid = ranked_chunks_to_docids(ranks, top_k=max(ks))
        docids_rerank = ranked_chunks_to_docids(reranked_chunks, top_k=final_top_k)

        for k in ks:
            hits_hybrid[k] += bool(set(docids_hybrid[:k]) & gold)
            hits_rerank[k] += bool(set(docids_rerank[:k]) & gold)

        f2_hybrid_sum += precision_recall_f2(docids_hybrid[:10], gold)[2]
        f2_rerank_sum += precision_recall_f2(docids_rerank[:10], gold)[2]

    print("\n=== Hybrid (Best Alpha) ===")
    for k in ks:
        print(f"  Hit@{k}: {hits_hybrid[k]/n*100:.2f}%")
    print(f"  F2@10: {f2_hybrid_sum/n:.4f}")

    print("\n=== Hybrid + Reranker ===")
    for k in ks:
        print(f"  Hit@{k}: {hits_rerank[k]/n*100:.2f}%")
    print(f"  F2@10: {f2_rerank_sum/n:.4f}")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [11]:
best_alpha, best_hybrid_ranks = evaluate_alpha_batched(
    questions=questions_train,
    golds_docid_sets=golds_docid_sets,
    alphas=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
    question_chunk_size=200,
    hybrid_top_n=50
)

evaluate_reranker(
    questions=questions_train,
    hybrid_chunk_ranks=best_hybrid_ranks,
    golds_docid_sets=golds_docid_sets
)

Đang tính Hybrid Search theo lô (200 câu/lần)...


Question chunks: 100%|██████████| 35/35 [2:28:01<00:00, 253.74s/it]



Alpha         Hit@1      Hit@5     Hit@10      F2@10
0.0          49.70%     78.77%     84.99%    0.3181
0.1          53.37%     81.17%     86.83%    0.3269
0.2          57.00%     83.31%     88.26%    0.3340
0.3          59.17%     84.77%     89.19%    0.3389
0.4          61.14%     85.64%     89.41%    0.3410
0.5          62.81%     85.91%     89.51%    0.3417

>>> Alpha tốt nhất theo F2@10: alpha = 0.5 (F2 = 0.3417)
Đang chuẩn bị cặp để rerank...
Tổng số cặp cần rerank: 350000


Batches:   0%|          | 0/5469 [00:00<?, ?it/s]


=== Hybrid (Best Alpha) ===
  Hit@1: 62.81%
  Hit@5: 85.91%
  Hit@10: 89.51%
  F2@10: 0.3417

=== Hybrid + Reranker ===
  Hit@1: 62.51%
  Hit@5: 84.41%
  Hit@10: 88.09%
  F2@10: 0.3368
